In [1]:
import re
from sentence_transformers import SentenceTransformer
from torch.utils.data import DataLoader, Dataset
import torch
import torch.nn as nn
import torch.optim as optim

/home/mayur/.local/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def parse_input_file(file_path):
    anchor, pros, cons = "", [], []
    with open(file_path, "r", encoding="utf-8") as file:
        lines = file.readlines()
    for line in lines:
        line = line.strip()
        if line.startswith("Discussion Title:"):
            anchor = line.split("Discussion Title:", 1)[1].strip()
        elif "Pro:" in line:
            clean_line = re.sub(r"\[.*?\]\(.*?\)", "", line.split("Pro:", 1)[1].strip())
            pros.append(clean_line)
        elif "Con:" in line:
            clean_line = re.sub(r"\[.*?\]\(.*?\)", "", line.split("Con:", 1)[1].strip())
            cons.append(clean_line)
    return anchor, pros, cons

file_path = "./Data/File1.txt"
anchor, pros, cons = parse_input_file(file_path)

pairs = []
for pro in pros:
    pairs.append({"first": anchor, "second": pro, "label": 1})
for pro in pros:
    for con in cons:
        pairs.append({"first": pro, "second": con, "label": 0})


In [3]:
from sentence_transformers import InputExample

# Convert the dataset into InputExample format
def prepare_dataset(data):
    examples = []
    for pair in data:
        examples.append(InputExample(
            texts=[pair['first'], pair['second']],
            label=float(pair['label'])
        ))
    return examples

In [4]:
train_examples = prepare_dataset(pairs)

In [5]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(train_examples, shuffle=True, batch_size=16)

In [ ]:
from sentence_transformers import SentenceTransformer, losses

model = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')
train_loss = losses.ContrastiveLoss(
    model=model,
    margin=0.5,
    distance_metric=losses.SiameseDistanceMetric.COSINE_DISTANCE
)

model.fit(
    train_objectives=[(train_dataloader, train_loss)],
    epochs=3,
    warmup_steps=100,
    output_path='./siamese-mpnet-kialo'
)